In [ ]:
! pip install -U \
  langgraph \
  langgraph-checkpoint-postgres \
  psycopg[binary,pool] \
  langchain-openai

In [1]:
import os

from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

/home/md-muktaruzzaman-anim/Desktop/ai-product-engineering-track/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


True

In [2]:
# Environment variables

DB_URI = os.getenv("DB_URI")

if not DB_URI:
    raise ValueError("DB_URI is not set in .env")


# Model

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
# Node
def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [4]:
# Build graph
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [5]:
# PostgreSQL Checkpointer
# =========================

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:

    # Run once to create LangGraph checkpoint tables
    checkpointer.setup()

    graph = builder.compile(checkpointer=checkpointer)

    # Thread 1
    t1 = {
        "configurable": {
            "thread_id": "thread-1"
        }
    }

    graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Hi, my name is Muktar"
                }
            ]
        },
        t1
    )

    out1 = graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What is my name?"
                }
            ]
        },
        t1
    )

    print("Thread-1:", out1["messages"][-1].content)

Thread-1: Your name is Muktar. How can I help you today?


In [6]:
with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    # Run ONCE (creates tables)
    checkpointer.setup()

    graph = builder.compile(checkpointer=checkpointer)

    # Thread 2 (fresh)
    t2 = {"configurable": {"thread_id": "thread-2"}}
    out2 = graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, t2)
    print("Thread-2:", out2["messages"][-1].content)


Thread-2: I can't determine your name unless you tell me. If you share it with me, I can use it in our conversation!


In [7]:
from langgraph.checkpoint.postgres import PostgresSaver

t1 = {"configurable": {"thread_id": "thread-1"}}

with PostgresSaver.from_conn_string(DB_URI) as cp:
    g = builder.compile(checkpointer=cp)

    snap = g.get_state(t1)  # <-- pulls from Postgres
    msgs = snap.values.get("messages", [])
    print("Last message:", msgs[-1].content if msgs else None)


Last message: Your name is Muktar. How can I help you today?
